## Prosta analiza obrazu termowizyjnego

In [16]:
import cv2
import numpy as np

In [17]:
# 1. Wczytanie sekwencji wideo
cap = cv2.VideoCapture('vid1_IR.avi')

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # 2. Konwersja do skali szarości (dla IR zazwyczaj już grayscale, ale dla bezpieczeństwa)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 4. Filtracja – medianowy filtr + operacje morfologiczne (usuwa szumy, łączy kontury)
    binary = cv2.medianBlur(binary, 5)

    # 3. Binaryzacja – próg dobrany eksperymentalnie
    _, binary = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=3)

    # 5. Indeksacja – connected components with stats
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary, connectivity=8)

    # 6. Analiza wyników indeksacji
    for i in range(1, num_labels):  # Pomijamy tło (label 0)
        x, y, w, h, area = stats[i]

        # a) Rysowanie prostokątów otaczających
        # b) Filtrowanie po rozmiarze
        if area < 500:
            continue

        # c) Filtrowanie po proporcjach – szukamy sylwetek (wysokie, wąskie)
        ratio = h / w if w > 0 else 0
        if ratio < 1.5:
            continue

        # d) Heurystyka łączenia sylwetek (bardzo uproszczona):
        # Zamiast łaczenia prostokątów, tylko zaznaczamy. Łączenie wymaga osobnej logiki np. sortowania po y i sprawdzania odległości.

        # Rysowanie końcowego prostokąta
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Pokazanie obrazu
    cv2.imshow('Termowizja - Detekcja', frame)
    cv2.imshow('Binaryzacja', binary)

    # Zakończenie po naciśnięciu 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
